# Audio Feature Extraction
Dieses Notebook extrahiert pro WAV-Datei in `data/processed/audio/` eine Reihe akustischer Merkmale und schreibt die aggregierten Werte nach `data/features/audio_features.csv`.

Extrahierte Features (pro Datei):
- `bpm` (Tempo in Beats Per Minute)
- `rms_mean`, `rms_std` (Lautheit / Energie)
- `spectral_centroid_mean`, `spectral_centroid_std` (Klangfarbe / Helligkeit)
- `spectral_bandwidth_mean`, `spectral_bandwidth_std` (Frequenzstreuung)
- `chroma_var_0` ... `chroma_var_11` (Varianz der 12 Chroma-Bins)
- `speech_ratio` (optional, Anteil gesprochener Frames; benötigt `webrtcvad`)

Hinweis: Das Notebook lädt Audios mit `librosa.load(..., sr=None)` (Original-SR). `webrtcvad` ist optional — wenn nicht installiert, bleibt `speech_ratio` NaN.

In [4]:
# Imports und Setup
from pathlib import Path
import glob
import logging
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

# Robustly locate repository root (walk upwards looking for common markers)
def find_repo_root(start: Path = None):
    start = Path.cwd() if start is None else Path(start)
    markers = ['tiktok_100_vs_100', 'requirements.txt', 'README.md', '.git']
    for p in [start] + list(start.parents):
        for m in markers:
            if (p / m).exists():
                return p
    return start

repo_root = find_repo_root()
# Data, output and log dirs (repo-root based)
DATA_AUDIO_DIR = repo_root / 'data' / 'processed' / 'audio'
OUTPUT_DIR = repo_root / 'features'
LOG_DIR = repo_root / 'audio_Analyse' / 'logs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Logger: mirror the pattern used in 01_Audio_Extraction (File + Stream handler)
log_file = LOG_DIR / 'feature_extraction.log'
logger = logging.getLogger('audio_features')
logger.setLevel(logging.INFO)
# Clear existing handlers to avoid duplicates in notebooks
if logger.hasHandlers():
    logger.handlers.clear()
fh = logging.FileHandler(log_file, mode='a', encoding='utf-8')
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch = logging.StreamHandler()
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)

logger.info(f'Repo root: {repo_root}')
logger.info(f'Data audio dir: {DATA_AUDIO_DIR}')
logger.info(f'Output dir: {OUTPUT_DIR}')
logger.info(f'Log file: {log_file}')

2025-11-06 09:35:48,869 - INFO - Repo root: c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse
2025-11-06 09:35:48,871 - INFO - Data audio dir: c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\data\processed\audio
2025-11-06 09:35:48,872 - INFO - Output dir: c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\features
2025-11-06 09:35:48,873 - INFO - Log file: c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\audio_Analyse\logs\feature_extraction.log
2025-11-06 09:35:48,871 - INFO - Data audio dir: c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\data\processed\audio
2025-11-06 09:35:48,872 - INFO - Output dir: c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\features
2025-11-06 09:35:48,873 - INFO - Log file: c:\Users\CelinaB\Docum

In [5]:
# Optional: webrtcvad prüfen und Speech-Ratio Funktion definieren
try:
    import webrtcvad
    VAD_AVAILABLE = True
    logger.info('webrtcvad is available; speech_ratio will be computed')
except Exception:
    webrtcvad = None
    VAD_AVAILABLE = False
    logger.info('webrtcvad not installed; speech_ratio will be NaN')

def compute_speech_ratio(y, sr, frame_ms=30, vad_mode=2):
    """Berechnet den Anteil von Frames, die webrtcvad als Sprache klassifiziert.
    Gibt np.nan zurück, wenn webrtcvad nicht verfügbar ist oder keine Frames vorhanden sind.
    """
    if not VAD_AVAILABLE:
        return np.nan
    # webrtcvad benötigt 16-bit PCM bei einer unterstützten SR; wir nutzen 16000 Hz
    target_sr = 16000
    if sr != target_sr:
        y_rs = librosa.resample(y, orig_sr=sr, target_sr=target_sr)
    else:
        y_rs = y
    pcm16 = (np.clip(y_rs, -1.0, 1.0) * 32767).astype(np.int16)
    raw = pcm16.tobytes()
    vad = webrtcvad.Vad(vad_mode)
    frame_bytes = int(target_sr * (frame_ms / 1000.0) * 2)
    n_frames = len(raw) // frame_bytes
    if n_frames == 0:
        return np.nan
    speech = 0
    for i in range(n_frames):
        s = i * frame_bytes
        e = s + frame_bytes
        if vad.is_speech(raw[s:e], sample_rate=target_sr):
            speech += 1
    return float(speech) / float(n_frames)

2025-11-06 09:35:48,893 - INFO - webrtcvad is available; speech_ratio will be computed


In [6]:
# Haupt-Extraktion: WAVs laden, Features berechnen, speichern
wav_paths = sorted(glob.glob(str(DATA_AUDIO_DIR / '*.wav')))  # alle .wav Dateien im Ordner
logger.info(f'Found {len(wav_paths)} wav files')
rows = []
for p in tqdm(wav_paths, desc='Extracting audio features'):
    vid = Path(p).stem
    try:
        y, sr = librosa.load(p, sr=None, mono=False)
        # Falls Stereo, mixe zu Mono
        if isinstance(y, np.ndarray) and y.ndim > 1:
            y = librosa.to_mono(y)
        y = y.astype(float)

        # BPM
        try:
            tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
            bpm = float(tempo)
        except Exception as e:
            logger.warning(f'BPM estimation failed for {vid}: {e}')
            bpm = np.nan

        # RMS
        try:
            rms = librosa.feature.rms(y=y)
            rms_mean = float(np.mean(rms))
            rms_std = float(np.std(rms))
        except Exception as e:
            logger.warning(f'RMS failed for {vid}: {e}')
            rms_mean = np.nan; rms_std = np.nan

        # Spectral centroid
        try:
            sc = librosa.feature.spectral_centroid(y=y, sr=sr)
            spectral_centroid_mean = float(np.mean(sc))
            spectral_centroid_std = float(np.std(sc))
        except Exception as e:
            logger.warning(f'Spectral centroid failed for {vid}: {e}')
            spectral_centroid_mean = np.nan; spectral_centroid_std = np.nan

        # Spectral bandwidth
        try:
            sb = librosa.feature.spectral_bandwidth(y=y, sr=sr)
            spectral_bandwidth_mean = float(np.mean(sb))
            spectral_bandwidth_std = float(np.std(sb))
        except Exception as e:
            logger.warning(f'Spectral bandwidth failed for {vid}: {e}')
            spectral_bandwidth_mean = np.nan; spectral_bandwidth_std = np.nan

        # Chroma variance (12 bins)
        try:
            chroma = librosa.feature.chroma_stft(y=y, sr=sr)
            chroma_var = np.var(chroma, axis=1)
        except Exception as e:
            logger.warning(f'Chroma failed for {vid}: {e}')
            chroma_var = np.array([np.nan]*12)

        # Optional speech ratio
        try:
            speech_ratio = compute_speech_ratio(y, sr)
        except Exception as e:
            logger.warning(f'Speech-ratio failed for {vid}: {e}')
            speech_ratio = np.nan

        row = dict(
            video_id=vid, bpm=bpm,
            rms_mean=rms_mean, rms_std=rms_std,
            spectral_centroid_mean=spectral_centroid_mean, spectral_centroid_std=spectral_centroid_std,
            spectral_bandwidth_mean=spectral_bandwidth_mean, spectral_bandwidth_std=spectral_bandwidth_std,
            speech_ratio=speech_ratio
        )
        for i, v in enumerate(chroma_var):
            row[f'chroma_var_{i}'] = float(v) if not np.isnan(v) else np.nan
        rows.append(row)

    except Exception as e:
        logger.exception(f'Failed processing {p}: {e}')
        continue

# Ergebnis DataFrame und Export
df = pd.DataFrame(rows)
out_file = OUTPUT_DIR / 'audio_features.csv'
df.to_csv(out_file, index=False)
logger.info(f'Wrote {len(df)} rows to {out_file}')
df.head()

2025-11-06 09:35:48,916 - INFO - Found 197 wav files
Extracting audio features:   0%|          | 0/197 [00:00<?, ?it/s]C:\Users\CelinaB\AppData\Local\Temp\ipykernel_41928\1501767981.py:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  bpm = float(tempo)
C:\Users\CelinaB\AppData\Local\Temp\ipykernel_41928\1501767981.py:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  bpm = float(tempo)
Extracting audio features: 100%|██████████| 197/197 [02:18<00:00,  1.42it/s]
2025-11-06 09:38:07,717 - INFO - Wrote 197 rows to c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\features\audio_features.csv

2025-11

,video_id,bpm,rms_mean,rms_std,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,speech_ratio,chroma_var_0,...,chroma_var_2,chroma_var_3,chroma_var_4,chroma_var_5,chroma_var_6,chroma_var_7,chroma_var_8,chroma_var_9,chroma_var_10,chroma_var_11
0,normal_100_likes_22500_id_7516938565066951991,139.674831,0.014903,0.019588,2816.768409,1743.759460,3153.345758,840.977366,0.908403,0.105630,...,0.106298,0.091258,0.089877,0.074280,0.078157,0.089455,0.094581,0.088492,0.105486,0.122652
1,normal_10_likes_83700_id_7546779048593181965,67.999589,0.025423,0.015674,4474.662061,1908.788674,4091.647865,722.189966,0.994350,0.104571,...,0.099630,0.117152,0.135053,0.123853,0.106651,0.107584,0.094656,0.094529,0.088848,0.089647
2,normal_11_likes_30500_id_7559969376590515470,107.666016,0.126814,0.057115,3212.963612,1091.773264,3643.004116,753.319422,0.995717,0.074363,...,0.077808,0.081846,0.083621,0.081858,0.077154,0.078561,0.078145,0.080191,0.076563,0.079554
3,normal_12_likes_51000_id_7557068031394991415,135.999178,0.044645,0.052302,3050.954201,1684.081678,3179.355446,979.727218,0.955752,0.106044,...,0.100279,0.112640,0.104974,0.105823,0.096001,0.092091,0.103063,0.114339,0.100204,0.104158
4,normal_13_likes_14100_id_7564088602234309918,126.048018,0.174276,0.115863,3562.231201,1881.436512,3370.279374,736.712843,0.989597,0.116176,...,0.089310,0.069206,0.059550,0.087237,0.084836,0.079574,0.072699,0.072038,0.076815,0.091000


## How to run
1. Stelle sicher, dass die benötigten Pakete installiert sind (siehe `requirements.txt`). Optional: `pip install webrtcvad`.
2. Öffne dieses Notebook in Jupyter Lab/Notebook und führe die Zellen der Reihe nach aus.
3. Das Ergebnis wird in `features/audio_features.csv` geschrieben.